In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv


In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# ================================
# Load Dataset
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

month_cols = df.columns[1:]   # Month columns

X = df[month_cols].values

# Target: failure in last month
y = (df[month_cols[-1]] > 0).astype(int).values

print("Class Distribution:", np.bincount(y))
print()

# ================================
# PNN
# ================================
class PNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x - xi) ** 2) / (2 * self.sigma ** 2))

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)

    def predict(self, X):
        preds = []
        for x in X:
            scores = []
            for c in self.classes:
                Xc = self.X[self.y == c]
                g = np.sum([self._gauss(x, xi) for xi in Xc])
                scores.append(g)
            preds.append(self.classes[np.argmax(scores)])
        return np.array(preds)


# ================================
# GRNN
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x - xi) ** 2) / (2 * self.sigma ** 2))

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            w = np.array([self._gauss(x, xi) for xi in self.X])
            if np.sum(w) == 0:
                preds.append(0)
            else:
                y_hat = np.sum(w * self.y) / np.sum(w)
                preds.append(int(round(y_hat)))
        return np.array(preds)


# ================================
# Evaluation Function
# ================================
def evaluate(model, X, y, name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        yp = model.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp, zero_division=0))
        rec.append(recall_score(yte, yp, zero_division=0))
        f1.append(f1_score(yte, yp, zero_division=0))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()


# ================================
# Models
# ================================
models = [
    (PNN(0.5), "PNN"),
    (GRNN(0.5), "GRNN"),
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
]

# ================================
# Run
# ================================
for m, name in models:
    evaluate(m, X, y, name)


Class Distribution: [ 0 48]

PNN
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0

GRNN
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0

KNN
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0

Decision Tree
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0



ValueError: The number of classes has to be greater than one; got 1 class

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# ================================
# Load Dataset
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

month_cols = df.columns[1:]   # Month columns

X = df[month_cols].values

# Target: failure in last month
y = (df[month_cols[-1]] > 0).astype(int).values

print("Class Distribution:", np.bincount(y))
print()

# ================================
# PNN
# ================================
class PNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x - xi) ** 2) / (2 * self.sigma ** 2))

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)

    def predict(self, X):
        preds = []
        for x in X:
            scores = []
            for c in self.classes:
                Xc = self.X[self.y == c]
                g = np.sum([self._gauss(x, xi) for xi in Xc])
                scores.append(g)
            preds.append(self.classes[np.argmax(scores)])
        return np.array(preds)


# ================================
# GRNN
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x - xi) ** 2) / (2 * self.sigma ** 2))

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            w = np.array([self._gauss(x, xi) for xi in self.X])
            if np.sum(w) == 0:
                preds.append(0)
            else:
                y_hat = np.sum(w * self.y) / np.sum(w)
                preds.append(int(round(y_hat)))
        return np.array(preds)


# ================================
# Evaluation Function
# ================================
def evaluate(model, X, y, name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        yp = model.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp, zero_division=0))
        rec.append(recall_score(yte, yp, zero_division=0))
        f1.append(f1_score(yte, yp, zero_division=0))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()


# ================================
# Models
# ================================
models = [
    (PNN(0.5), "PNN"),
    (GRNN(0.5), "GRNN"),
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
]

# ================================
# Run
# ================================
for m, name in models:
    evaluate(m, X, y, name)


In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# ================================
# Load Dataset
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

X = df[month_cols[:-1]].values
y = (df[month_cols[-1]] > 0).astype(int).values

print("Class Distribution:", np.bincount(y))


class_counts = np.bincount(y)
print("Class Distribution:", class_counts)
print()

# ================================
# Decide Safe Number of Splits
# ================================
minority_class = np.min(class_counts)
n_splits = min(5, minority_class)   # NEVER exceed minority size

if n_splits < 2:
    raise ValueError("Not enough samples in minority class for cross-validation.")

print(f"Using {n_splits}-fold Stratified CV\n")

# ================================
# PNN
# ================================
class PNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x - xi) ** 2) / (2 * self.sigma ** 2))

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)

    def predict(self, X):
        preds = []
        for x in X:
            scores = []
            for c in self.classes:
                Xc = self.X[self.y == c]
                g = np.sum([self._gauss(x, xi) for xi in Xc])
                scores.append(g)
            preds.append(self.classes[np.argmax(scores)])
        return np.array(preds)

# ================================
# GRNN
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def _gauss(self, x, xi):
        return np.exp(-np.sum((x - xi) ** 2) / (2 * self.sigma ** 2))

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            w = np.array([self._gauss(x, xi) for xi in self.X])
            if np.sum(w) == 0:
                preds.append(0)
            else:
                y_hat = np.sum(w * self.y) / np.sum(w)
                preds.append(int(round(y_hat)))
        return np.array(preds)

# ================================
# Evaluation
# ================================
def evaluate(model, X, y, name):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        # If somehow only one class in training fold, skip
        if len(np.unique(ytr)) < 2:
            continue

        model.fit(Xtr, ytr)
        yp = model.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp, zero_division=0))
        rec.append(recall_score(yte, yp, zero_division=0))
        f1.append(f1_score(yte, yp, zero_division=0))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()

# ================================
# Models
# ================================
models = [
    (PNN(0.5), "PNN"),
    (GRNN(0.5), "GRNN"),
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
]

# ================================
# Run
# ================================
for m, name in models:
    evaluate(m, X, y, name)


Class Distribution: [ 0 48]
Class Distribution: [ 0 48]



ValueError: Not enough samples in minority class for cross-validation.

In [13]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# ================================
# Load Dataset
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

month_cols = df.columns[1:]

# 🔥 IMPORTANT: remove last month from features
X = df[month_cols[:-1]].values

# Target = failure in last month
y = (df[month_cols[-1]] > 0).astype(int).values

print("Class Distribution:", np.bincount(y))
print()

# ================================
# PNN (Improved)
# ================================
class PNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)

    def _gauss(self, x, Xc):
        diff = Xc - x
        return np.exp(-np.sum(diff**2, axis=1) / (2 * self.sigma**2))

    def predict(self, X):
        preds = []
        for x in X:
            scores = []
            for c in self.classes:
                Xc = self.X[self.y == c]
                g = np.sum(self._gauss(x, Xc)) / len(Xc)  # normalize
                scores.append(g)
            preds.append(self.classes[np.argmax(scores)])
        return np.array(preds)

# ================================
# GRNN (Improved)
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            diff = self.X - x
            w = np.exp(-np.sum(diff**2, axis=1) / (2 * self.sigma**2))
            s = np.sum(w)
            if s == 0:
                preds.append(0)
            else:
                y_hat = np.sum(w * self.y) / s
                preds.append(int(y_hat >= 0.5))
        return np.array(preds)

# ================================
# Evaluation Function
# ================================
def evaluate(model, X, y, name):
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        yp = model.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp, zero_division=0))
        rec.append(recall_score(yte, yp, zero_division=0))
        f1.append(f1_score(yte, yp, zero_division=0))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()

# ================================
# Models
# ================================
models = [
    (PNN(0.5), "PNN"),
    (GRNN(0.5), "GRNN"),
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
]

# ================================
# Run
# ================================
for m, name in models:
    evaluate(m, X, y, name)


Class Distribution: [ 0 48]

PNN
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0

GRNN
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0

KNN
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0

Decision Tree
  Accuracy : 1.0
  Precision: 1.0
  Recall   : 1.0
  F1       : 1.0



ValueError: The number of classes has to be greater than one; got 1 class

In [14]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# ================================
# Load Dataset
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

month_cols = df.columns[1:]

# Features = all months except last
X = df[month_cols[:-1]].values

# Original target
last_month = df[month_cols[-1]].values

# ================================
# FIX: Handle One-Class Problem
# ================================
y = (last_month > 0).astype(int)

if len(np.unique(y)) < 2:
    print("⚠ Only one class detected. Creating balanced target using median threshold.")
    threshold = np.median(last_month)
    y = (last_month > threshold).astype(int)

print("Final Class Distribution:", np.bincount(y))
print()

# ================================
# PNN
# ================================
class PNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)

    def predict(self, X):
        preds = []
        for x in X:
            scores = []
            for c in self.classes:
                Xc = self.X[self.y == c]
                diff = Xc - x
                g = np.exp(-np.sum(diff**2, axis=1) / (2 * self.sigma**2))
                scores.append(np.sum(g) / len(Xc))
            preds.append(self.classes[np.argmax(scores)])
        return np.array(preds)

# ================================
# GRNN
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            diff = self.X - x
            w = np.exp(-np.sum(diff**2, axis=1) / (2 * self.sigma**2))
            s = np.sum(w)
            if s == 0:
                preds.append(0)
            else:
                y_hat = np.sum(w * self.y) / s
                preds.append(int(y_hat >= 0.5))
        return np.array(preds)

# ================================
# Evaluation Function
# ================================
def evaluate(model, X, y, name):
    if len(np.unique(y)) < 2:
        print(f"{name} skipped (only one class present)\n")
        return

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        yp = model.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp, zero_division=0))
        rec.append(recall_score(yte, yp, zero_division=0))
        f1.append(f1_score(yte, yp, zero_division=0))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()

# ================================
# Models
# ================================
models = [
    (PNN(0.5), "PNN"),
    (GRNN(0.5), "GRNN"),
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
]

# ================================
# Run
# ================================
for m, name in models:
    evaluate(m, X, y, name)


⚠ Only one class detected. Creating balanced target using median threshold.
Final Class Distribution: [30 18]

PNN
  Accuracy : 0.4178
  Precision: 0.2467
  Recall   : 0.3333
  F1       : 0.2749

GRNN
  Accuracy : 0.3956
  Precision: 0.22
  Recall   : 0.2667
  F1       : 0.236

KNN
  Accuracy : 0.3533
  Precision: 0.1417
  Recall   : 0.2167
  F1       : 0.1667

Decision Tree
  Accuracy : 0.6089
  Precision: 0.5167
  Recall   : 0.6167
  F1       : 0.541

SVM
  Accuracy : 0.3533
  Precision: 0.1257
  Recall   : 0.2167
  F1       : 0.1591

Bagging
  Accuracy : 0.6711
  Precision: 0.6
  Recall   : 0.45
  F1       : 0.5067

Logistic Regression
  Accuracy : 0.4978
  Precision: 0.3267
  Recall   : 0.4333
  F1       : 0.3667



In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

# ================================
# Load NEW Dataset
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/2-data/blue_mountain_simulated_failure_times_corrected.csv"
df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print(df.head())

# ================================
# Feature / Target Split
# ================================
month_cols = df.columns[1:]

# Features = all except last column
X = df[month_cols[:-1]].values

# Target = last month
last_month = df[month_cols[-1]].values

# Convert to binary classification
y = (last_month > 0).astype(int)

# ================================
# Handle One-Class Problem
# ================================
if len(np.unique(y)) < 2:
    print("⚠ Only one class detected. Creating balanced target using median threshold.")
    threshold = np.median(last_month)
    y = (last_month > threshold).astype(int)

print("Final Class Distribution:", np.bincount(y))
print()

# ================================
# PNN
# ================================
class PNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.classes = np.unique(y)

    def predict(self, X):
        preds = []
        for x in X:
            scores = []
            for c in self.classes:
                Xc = self.X[self.y == c]
                diff = Xc - x
                g = np.exp(-np.sum(diff**2, axis=1) / (2 * self.sigma**2))
                scores.append(np.sum(g) / len(Xc))
            preds.append(self.classes[np.argmax(scores)])
        return np.array(preds)

# ================================
# GRNN
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = sigma

    def fit(self, X, y):
        self.X = X
        self.y = y

    def predict(self, X):
        preds = []
        for x in X:
            diff = self.X - x
            w = np.exp(-np.sum(diff**2, axis=1) / (2 * self.sigma**2))
            s = np.sum(w)
            if s == 0:
                preds.append(0)
            else:
                y_hat = np.sum(w * self.y) / s
                preds.append(int(y_hat >= 0.5))
        return np.array(preds)

# ================================
# Evaluation Function
# ================================
def evaluate(model, X, y, name):
    if len(np.unique(y)) < 2:
        print(f"{name} skipped (only one class present)\n")
        return

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)

        model.fit(Xtr, ytr)
        yp = model.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp, zero_division=0))
        rec.append(recall_score(yte, yp, zero_division=0))
        f1.append(f1_score(yte, yp, zero_division=0))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()

# ================================
# Models
# ================================
models = [
    (PNN(0.5), "PNN"),
    (GRNN(0.5), "GRNN"),
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000, class_weight="balanced"), "Logistic Regression")
]

# ================================
# Run
# ================================
for m, name in models:
    evaluate(m, X, y, name)


Dataset Shape: (1131, 3)
   SMP  FailureTimeDays  Failure_Time
0    1             4.74           NaN
1    1            20.23           NaN
2    1            22.21           NaN
3    1            26.02           NaN
4    1            35.09           NaN
Final Class Distribution: [948 183]

PNN
  Accuracy : 0.0
  Precision: 0.0
  Recall   : 0.0
  F1       : 0.0

GRNN
  Accuracy : 0.8382
  Precision: 0.0
  Recall   : 0.0
  F1       : 0.0



ValueError: Input X contains NaN.
KNeighborsClassifier does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [2]:
import numpy as np
import pandas as pd

file_path = "/kaggle/input/datasets/aayushwhizham/2-data/blue_mountain_simulated_failure_times_corrected.csv"
df = pd.read_csv(file_path)

print("Before Cleaning:")
print(df.head())
print(df.tail())

# Merge both columns into one
df["FailureTime"] = df["FailureTimeDays"].combine_first(df["Failure_Time"])

# Drop old columns
df = df.drop(columns=["FailureTimeDays", "Failure_Time"])

print("\nAfter Merging:")
print(df.head())
print(df.isna().sum())


Before Cleaning:
   SMP  FailureTimeDays  Failure_Time
0    1             4.74           NaN
1    1            20.23           NaN
2    1            22.21           NaN
3    1            26.02           NaN
4    1            35.09           NaN
      SMP  FailureTimeDays  Failure_Time
1126   45              NaN        352.72
1127   45              NaN        357.66
1128   45              NaN        396.49
1129   45              NaN        421.80
1130   45              NaN        446.03

After Merging:
   SMP  FailureTime
0    1         4.74
1    1        20.23
2    1        22.21
3    1        26.02
4    1        35.09
SMP            0
FailureTime    0
dtype: int64


In [3]:
threshold = df["FailureTime"].median()

y = (df["FailureTime"] < threshold).astype(int)
X = df[["FailureTime"]].values

print("Class Distribution:", np.bincount(y))

Class Distribution: [566 565]


In [6]:
y

array([5., 1., 3., 3., 1., 4., 4., 2., 2., 1., 1., 2., 1., 1., 2., 1., 3.,
       2., 2., 2., 5., 8., 2., 4., 1., 3., 5., 2., 1., 2., 2., 2., 2., 3.,
       3., 3., 2., 3., 1., 2., 1., 4., 3., 2., 2., 4., 2., 2.])

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

def evaluate(model, X, y, name):

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    acc, pre, rec, f1 = [], [], [], []

    for tr, te in skf.split(X, y):
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y[tr], y[te]

        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", model)
        ])

        pipeline.fit(Xtr, ytr)
        yp = pipeline.predict(Xte)

        acc.append(accuracy_score(yte, yp))
        pre.append(precision_score(yte, yp))
        rec.append(recall_score(yte, yp))
        f1.append(f1_score(yte, yp))

    print(f"{name}")
    print("  Accuracy :", round(np.mean(acc), 4))
    print("  Precision:", round(np.mean(pre), 4))
    print("  Recall   :", round(np.mean(rec), 4))
    print("  F1       :", round(np.mean(f1), 4))
    print()

models = [
    (KNeighborsClassifier(n_neighbors=5), "KNN"),
    (DecisionTreeClassifier(random_state=42), "Decision Tree"),
    (SVC(kernel="rbf", class_weight="balanced"), "SVM"),
    (BaggingClassifier(n_estimators=50, random_state=42), "Bagging"),
    (LogisticRegression(max_iter=1000), "Logistic Regression")
]

for m, name in models:
    evaluate(m, X, y, name)


KNN
  Accuracy : 0.9982
  Precision: 1.0
  Recall   : 0.9965
  F1       : 0.9982

Decision Tree
  Accuracy : 0.9973
  Precision: 0.9982
  Recall   : 0.9965
  F1       : 0.9973

SVM
  Accuracy : 0.9973
  Precision: 0.9982
  Recall   : 0.9965
  F1       : 0.9973

Bagging
  Accuracy : 0.9973
  Precision: 0.9982
  Recall   : 0.9965
  F1       : 0.9973

Logistic Regression
  Accuracy : 0.9982
  Precision: 1.0
  Recall   : 0.9965
  F1       : 0.9982



In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression

# ================================
# Kernel regressor (PNN-style) - Nadaraya-Watson
# ================================
class PNNRegressor:
    def __init__(self, sigma=1.0):
        self.sigma = float(sigma)

    def _gauss(self, x, xi):
        # squared euclidean distance
        d2 = np.sum((x - xi) ** 2)
        return np.exp(-d2 / (2 * (self.sigma ** 2)))

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y, dtype=float)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = np.empty((X.shape[0],), dtype=float)
        for i, x in enumerate(X):
            weights = np.array([self._gauss(x, xi) for xi in self.X])
            s = weights.sum()
            if s == 0:
                preds[i] = np.mean(self.y)  # fallback to global mean
            else:
                preds[i] = (weights @ self.y) / s
        return preds

# ================================
# GRNN (same idea for continuous target) - kept separate for clarity
# ================================
class GRNN:
    def __init__(self, sigma=1.0):
        self.sigma = float(sigma)

    def _gauss(self, x, xi):
        d2 = np.sum((x - xi) ** 2)
        return np.exp(-d2 / (2 * (self.sigma ** 2)))

    def fit(self, X, y):
        self.X = np.asarray(X, dtype=float)
        self.y = np.asarray(y, dtype=float)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = np.empty((X.shape[0],), dtype=float)
        for i, x in enumerate(X):
            w = np.array([self._gauss(x, xi) for xi in self.X])
            sw = w.sum()
            if sw == 0:
                preds[i] = np.mean(self.y)
            else:
                preds[i] = (w @ self.y) / sw
        return preds

# ================================
# Evaluation helper (regression metrics)
# ================================
def evaluate_regression(model, X, y, n_splits=10):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmses, maes, r2s = [], [], []

    for tr_idx, te_idx in kf.split(X):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te).astype(float)

        rmses.append(np.sqrt(mean_squared_error(y_te, y_pred)))
        maes.append(mean_absolute_error(y_te, y_pred))
        r2s.append(r2_score(y_te, y_pred))

    return {
        "RMSE": float(np.mean(rmses)),
        "MAE": float(np.mean(maes)),
        "R2": float(np.mean(r2s))
    }

# ================================
# Load dataset (edit path / target_col as needed)
# ================================
file_path = "/kaggle/input/datasets/aayushwhizham/reliabledd/blue_mountain_supercomputer_monthly_failures_proper.csv"
df = pd.read_csv(file_path)

# If your CSV has a device id column named 'SMP' (or similar), drop it
possible_id_cols = ["SMP", "Device", "ID", "Index"]
drop_cols = [c for c in df.columns if c in possible_id_cols]
df = df.drop(columns=drop_cols, errors='ignore')

# Choose which column to predict. By default we predict the last column (month) as numeric count.
target_col = df.columns[-1]   # change if you want to predict a different month
feature_cols = [c for c in df.columns if c != target_col]

# Convert to numeric if strings present
df[feature_cols + [target_col]] = df[feature_cols + [target_col]].apply(pd.to_numeric, errors='coerce')

# Drop rows with NaNs (or handle them as you prefer)
df = df.dropna(axis=0, subset=feature_cols + [target_col]).reset_index(drop=True)

X = df[feature_cols].values.astype(float)
y = df[target_col].values.astype(float)

print(f"Loaded {df.shape[0]} rows, features: {len(feature_cols)} columns -> target='{target_col}'")
print("Feature columns:", feature_cols)

# ================================
# Models (regression versions)
# ================================
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.svm import SVR

models = [
    (PNNRegressor(sigma=0.5), "PNNRegressor (kernel)"),
    (GRNN(sigma=0.5), "GRNN (kernel)"),
    (KNeighborsRegressor(n_neighbors=5), "KNN Regressor (k=5)"),
    (DecisionTreeRegressor(random_state=42), "Decision Tree Regressor"),
    (SVR(kernel="rbf", C=1.0, epsilon=0.1), "SVR (RBF)"),
    (BaggingRegressor(n_estimators=50, random_state=42), "Bagging Regressor (50)"),
    (LinearRegression(), "Linear Regression")
]

# ================================
# Run evaluation
# ================================
results = []
for model, name in models:
    try:
        res = evaluate_regression(model, X, y, n_splits=10)
        res["model"] = name
    except Exception as e:
        res = {"model": name, "error": str(e)}
    results.append(res)

res_df = pd.DataFrame(results).set_index("model")
print("\nCross-validated regression metrics (mean over folds):")
print(res_df[["RMSE", "MAE", "R2"]])


Loaded 48 rows, features: 14 columns -> target='15'
Feature columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']

Cross-validated regression metrics (mean over folds):
                             RMSE       MAE        R2
model                                                
PNNRegressor (kernel)    1.743056  1.347771 -1.480250
GRNN (kernel)            1.743056  1.347771 -1.480250
KNN Regressor (k=5)      1.342189  1.060000 -0.319398
Decision Tree Regressor  1.330613  1.025000 -0.804503
SVR (RBF)                1.309172  1.004494 -0.255958
Bagging Regressor (50)   1.128399  0.896400 -0.155645
Linear Regression        1.553477  1.145210 -1.300689
